Instalando dependências

In [2]:
%%capture
!pip install jupyter_bokeh
!pip install python-dotenv


Definindo a chave de API do provedor LLM

*   Recupera a chave de API pelo objeto *userdata*
*   Cria o arquivo .env com a chave recuperada


In [3]:
# Temporarily get the API key to create the .env file
from google.colab import userdata
import os

# Check if .env file already exists
if not os.path.exists('.env'):
    GOOGLE_API_KEY_VALUE = userdata.get('google')

    with open('.env', 'w') as f:
        if GOOGLE_API_KEY_VALUE is None or len(GOOGLE_API_KEY_VALUE) == 0:
            f.write('GEMINI_API_KEY=GOOGLE_API_KEY_VALUE')
        else:
            f.write(f'GEMINI_API_KEY={GOOGLE_API_KEY_VALUE}')

    print('`.env` file created with GEMINI_API_KEY.')
else:
    print('`.env` file already exists. Skipping creation.')

`.env` file already exists. Skipping creation.


In [4]:
from google import genai
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY')

# Check if the API key is the placeholder value
if GOOGLE_API_KEY == 'GOOGLE_API_KEY_VALUE':
    raise ValueError("Please replace 'GOOGLE_API_KEY_VALUE' in your .env file with your actual GEMINI_API_KEY.")
elif GOOGLE_API_KEY is None or len(GOOGLE_API_KEY) == 0:
    raise ValueError("GEMINI_API_KEY not found in .env file or is empty. Please ensure it's set correctly.")

client = genai.Client(api_key=GOOGLE_API_KEY)

In [5]:
# Create .gitignore file
with open('.gitignore', 'w') as f:
    f.write('/.env\n')
    f.write('/.ipynb_checkpoints/\n')
print('`.gitignore` file created.')

`.gitignore` file created.


In [6]:
# Create env.example file
with open('env.example', 'w') as f:
    f.write('GEMINI_API_KEY=')
print('`env.example` file created.')

`env.example` file created.


### Configuração do Bot Especialista

Funções configuradas para o bot especialista:

1.  **Responda sem alucinar**:** Baseado exclusivamente em um contexto específico que você fornecerá.
2.  **Limite de 3 perguntas**:** O bot responderá a apenas três perguntas.
3.  **Resumo e Encerramento**:** Após a terceira resposta, o bot fará um resumo da conversa e encerrará a interação.

Para isso, você precisará definir o `EXPERT_CONTEXT` com as informações que seu bot deve usar. Abaixo, estou fornecendo um exemplo de contexto sobre um software fictício. **Por favor, substitua este texto pelo seu próprio contexto especializado.**

In [7]:
import panel as pn
pn.extension()

# --- Variáveis Globais de Estado do Chat ---
# Inicializa o contexto da conversa com uma mensagem de sistema que define a persona e as restrições do bot.
# Esta instrução é crucial para que o bot "não alucine" e se mantenha no contexto.
context = [
    {
        'role': 'system',
        'content': 'Você é um bot especialista em um tópico específico. Suas respostas devem ser baseadas EXCLUSIVAMENTE no contexto fornecido a cada interação. Se uma pergunta estiver fora deste contexto ou for genérica, responda educadamente que não pode fornecer informações sobre isso, pois você é um bot especialista e só pode usar o conhecimento que lhe foi dado.'
    }
]
qa_counter = 0
MAX_QUESTIONS = 4

# --- Definição do Contexto Especializado ---
# Este é o bloco onde você deve inserir as informações não públicas ou específicas do seu trabalho.
# O bot usará SOMENTE estas informações para responder. SUBSTITUA ESTE TEXTO!
EXPERT_CONTEXT = """
A empresa fictícia "FitTech Soluções Inteligentes Ltda." desenvolveu uma plataforma de gestão e acompanhamento de treinos chamada "GymAI", que utiliza Inteligência Artificial Generativa para auxiliar alunos, personal trainers e gestores de academias na criação, acompanhamento e adaptação de treinos.

Principais Funcionalidades do GymAI:

Criação Inteligente de Treinos: A IA gera sugestões de treinos personalizados com base em objetivos do aluno, nível de experiência, frequência semanal, equipamentos disponíveis e preferências de exercícios.
Adaptação de Treinos: A IA pode sugerir alterações no treinamento conforme a evolução do aluno, disponibilidade de equipamentos ou dificuldades relatadas durante os exercícios.
Acompanhamento de Desempenho: Registro de cargas, repetições, séries e frequência, permitindo que a IA identifique padrões de evolução e gere relatórios sobre o desempenho do aluno.
Assistente Virtual de Musculação: Chat baseado em IA generativa capaz de responder dúvidas sobre execução de exercícios, organização dos treinos, descanso, progressão de cargas e conceitos básicos de musculação.
Planejamento de Rotina: Organização dos treinos ao longo da semana, considerando disponibilidade do aluno, grupos musculares trabalhados e períodos de recuperação.

Benefícios para o Cliente:

Maior personalização dos treinos de acordo com as características e objetivos de cada aluno.
Redução do tempo necessário para que profissionais elaborem e ajustem rotinas de treinamento.
Maior acompanhamento da evolução dos alunos.
Facilidade para adaptar os treinos conforme mudanças na rotina ou disponibilidade de equipamentos.
Maior interação dos alunos com a academia por meio do assistente virtual baseado em IA.
Centralização das informações de treinamento e desempenho em uma única plataforma.

Requisitos Técnicos:

Acesso via navegador web e aplicativo para dispositivos móveis.
Solução baseada em SaaS (Software as a Service), sem necessidade de instalação de servidores locais.
Utilização de modelos de IA generativa para interpretação das informações fornecidas pelo usuário e geração das recomendações.
Armazenamento seguro dos dados dos usuários.
Sistema capaz de integrar informações de exercícios, treinos e histórico de desempenho.

Opções de Suporte:

Plano Básico: Suporte via e-mail, com resposta em até 24 horas úteis.
Plano Premium: Suporte prioritário 24/7 via chat e telefone, além de recursos avançados de personalização e acompanhamento.

Observações Importantes:

O GymAI foi desenvolvido para academias, personal trainers e praticantes de musculação.
As sugestões geradas pela IA possuem caráter informativo e de apoio, não substituindo a avaliação ou orientação de um profissional de educação física ou de saúde.
A IA utiliza os dados fornecidos pelo usuário para gerar recomendações, podendo existir limitações ou imprecisões nas respostas.
O sistema possui foco em treinamento físico e acompanhamento de desempenho, não sendo destinado ao diagnóstico ou tratamento de doenças.
O profissional de educação física pode revisar e ajustar as sugestões geradas pela IA antes de disponibilizá-las ao aluno.
"""

# --- Configuração da Interface de Chat (Panel) ---
# O widget de entrada de texto para o usuário.
inp = pn.widgets.TextInput(value="Olá", placeholder="Digite sua pergunta aqui...")
# O botão para enviar a mensagem.
button_conversation = pn.widgets.Button(name="Enviar")
# Uma lista para armazenar os painéis de chat (mensagens).
panels = []

# A função `collect_messages` será chamada quando o botão for clicado.
# Ela será definida em uma célula separada, mas já a preparamos para o binding.
# Esta linha é importante para que o botão acione a função quando definida.
# Removido o placeholder interactive_conversation e a definição do dashboard aqui.

In [8]:
from google.genai import types

def get_completion(prompt, temperature=0, top_p=0.95, top_k=20):
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
        ),
    )
    return response.text

In [9]:
from google.genai import types

def get_completion_from_messages(messages, model="gemini-3.6-flash", temperature=0):
    # Convert messages from the chat history format to Google GenAI format
    formatted_contents = []
    for msg in messages:
        role = msg['role']
        content = msg['content']
        # Gemini models use 'model' for the AI's role
        genai_role = 'user' if role == 'user' else 'model'
        formatted_contents.append({
            'role': genai_role,
            'parts': [{'text': content}]
        })

    response = client.models.generate_content(
        model=model,
        contents=formatted_contents,
        config=types.GenerateContentConfig(
            temperature=temperature,
        ),
    )
    return response.text

In [10]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))

    return pn.Column(*panels)

In [11]:
def collect_messages(_):
    global qa_counter, context # Garante que essas variáveis são globais

    # Se o limite de perguntas já foi atingido, apenas exibe uma mensagem e desabilita a entrada.
    if qa_counter >= MAX_QUESTIONS:
        if not inp.disabled:
            panels.append(pn.Row('Sistema:', pn.pane.Markdown("A conversa foi encerrada, pois o limite de perguntas foi atingido.", width=600, styles={'background-color': '#F0F8FF', 'border-radius': '8px', 'padding': '10px', 'border': '1px solid #C0D3E8', 'margin-bottom': '8px'})))
            inp.disabled = True
        return pn.Column(*panels)

    user_prompt_text = inp.value_input
    inp.value = ''  # Limpa o campo de entrada imediatamente

    # Adiciona a pergunta original do usuário aos painéis para exibição.
    panels.append(pn.Row('Usuário:', pn.pane.Markdown(user_prompt_text, width=600, styles={'background-color': '#EAEAEA', 'border-radius': '8px', 'padding': '10px', 'border': '1px solid #B8B8B8', 'margin-bottom': '8px'})))

    # Constrói a mensagem para o LLM, incluindo o contexto especializado e a pergunta do usuário.
    # Isso garante que o LLM use APENAS o `EXPERT_CONTEXT`.
    current_user_message_for_llm = {
        'role': 'user',
        'content': f"""Informações de contexto para o especialista (USE SOMENTE ESTAS INFORMAÇÕES):
---
{EXPERT_CONTEXT}
---

Pergunta do Usuário: {user_prompt_text}"""
    }

    # Envia o histórico completo da conversa (incluindo a instrução do sistema e a nova pergunta contextualizada) para o LLM.
    messages_to_send = context + [current_user_message_for_llm]

    response_from_llm = get_completion_from_messages(messages_to_send)

    # Incrementa o contador de perguntas após receber uma resposta.
    qa_counter += 1

    # Atualiza o contexto global da conversa com a pergunta original do usuário e a resposta do LLM.
    # Este `context` é usado para as próximas interações do `get_completion_from_messages`.
    context.append({'role':'user', 'content':user_prompt_text})
    context.append({'role':'assistant', 'content':response_from_llm})

    # Exibe a resposta do assistente nos painéis com um estilo.
    panels.append(pn.Row('Assistente:', pn.pane.Markdown(response_from_llm, width=600, styles={'background-color': '#E6E6FA', 'border-radius': '8px', 'padding': '10px', 'border': '1px solid #C6C6E0', 'margin-bottom': '8px'})))

    # --- Lógica de Resumo e Encerramento ---
    if qa_counter >= MAX_QUESTIONS:
        # Prepara a mensagem para gerar o resumo, enviando todo o histórico da conversa.
        summary_request_messages = context + [{'role':'user', 'content': "Por favor, forneça um breve resumo das perguntas e respostas desta conversa e, em seguida, declare claramente 'A conversa foi encerrada.'."}]
        summary_response = get_completion_from_messages(summary_request_messages)

        panels.append(pn.Row('Resumo Final:', pn.pane.Markdown(summary_response, width=600, styles={'background-color': '#D3D3D3', 'border-radius': '8px', 'padding': '10px', 'border': '1px solid #B3B3B3', 'margin-bottom': '8px'})))

        # Desabilita a entrada de texto para encerrar a conversa.
        inp.disabled = True
        panels.append(pn.Row('Sistema:', pn.pane.Markdown("O limite de perguntas foi atingido e a conversa foi encerrada. Você pode reiniciar o kernel para iniciar uma nova conversa.", width=600, styles={'background-color': '#F0F8FF', 'border-radius': '8px', 'padding': '10px', 'border': '1px solid #C0D3E8', 'margin-bottom': '8px'})))

    return pn.Column(*panels)

# Bind the button to the collect_messages function
interactive_conversation = pn.bind(collect_messages, button_conversation)

# Define and display the dashboard
dashboard = pn.Column(
    "## Bot Especialista: AgileFlow",
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=500),
)

dashboard

Column
    [0] Markdown(str)
    [1] TextInput(placeholder='Digite sua pergunta a...)
    [2] Row
        [0] Button(label='Enviar', name='Enviar')
    [3] ParamFunction(function, _pane=Column, defer_load=False, height=500, loading_indicator=True)